# Web scraping stock market news for Sentiment Analysis

## 1. Introduction

Stock market news articles from 2014-2021 will be collected by dynamic web scraping from [Investing.com](https://uk.investing.com/equities/astrazeneca-news) using a combination of Selenium library to automate browser interaction enabling data extraction by Beautiful Soup.




## 2. Install/import libraries

In [41]:
!pip install htmldate
!pip install twython
!pip3 install newspaper3k
!pip install lxml_html_clean

In [42]:
import pandas as pd
import numpy as np
import time
import twython
import requests
import nltk
import warnings
warnings.filterwarnings('ignore')

from htmldate import find_date
from tqdm import tqdm
from bs4 import BeautifulSoup
from nltk.sentiment.vader import SentimentIntensityAnalyzer
nltk.downloader.download('vader_lexicon')
from newspaper import Article

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\acer\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


## 3. Data collection



In [43]:
# Set up Selenium

!pip install undetected-chromedriver selenium webdriver_manager

# For Windows, you don't need the apt-get commands that are for Linux/Ubuntu
# Comment out or remove these Linux-specific commands:
# !apt-get update 
# !apt install chromium-chromedriver
# !cp /usr/lib/chromium-browser/chromedriver /usr/bin

# You need to download chromedriver for Windows separately and put it in your PATH
# or specify the full path to chromedriver.exe

import sys
# This path insert is not needed if you have chromedriver in your PATH or specify the full path
# sys.path.insert(0,'/usr/lib/chromium-browser/chromedriver')

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import time
import random
import numpy as np
from bs4 import BeautifulSoup
from selenium.webdriver.common.by import By
import undetected_chromedriver as uc

# ตั้งค่า options สำหรับ Chrome
chrome_options = uc.ChromeOptions()
# ไม่ใช้ headless เพื่อให้ดูเหมือนผู้ใช้จริง (ถ้าต้องการแสดงหน้าต่างจริง)
# chrome_options.add_argument('--headless')  
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')
# เปลี่ยน User-Agent ให้เหมือนกับเบราว์เซอร์จริง
chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36")
# ปิดการแจ้งเตือนว่าใช้ Selenium
chrome_options.add_argument("--disable-blink-features=AutomationControlled")

# สร้าง driver ด้วย undetected-chromedriver
driver = uc.Chrome(options=chrome_options)

In [44]:
from selenium.webdriver.common.by import By

def get_newslinks(company, page_number):
    """
    สำหรับ URL ที่ระบุ จะเลื่อนหน้าเว็บจนโหลดเนื้อหาทั้งหมด แล้วดึงลิงก์ข่าวของบริษัท
    :param company: ชื่อบริษัทที่ต้องการดึงข่าว (เช่น 'nvidia-corp')
    :param page_number: เลขหน้าของข่าวใน Investing.com
    :return: รายการ URL ของข่าวที่พบ
    """
    url = f"https://uk.investing.com/equities/{company}-news/{page_number}"
    driver.get(url)
    
    # รอให้หน้าโหลด (ใช้เวลาหน่วงแบบสุ่มเพื่อเลียนแบบผู้ใช้งานจริง)
    time.sleep(random.uniform(3, 5))
    
    # เลื่อนหน้าเว็บจนกว่าจะเลื่อนไม่ได้แล้ว (ช่วยให้ lazy load ทำงาน)
    old_position = 0
    new_position = None
    while new_position != old_position:
        old_position = driver.execute_script("return window.pageYOffset;")
        time.sleep(random.uniform(1, 2))
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        new_position = driver.execute_script("return window.pageYOffset;")
    
    cleaned_links = []
    try:
        # ค้นหาทุกบทความในส่วนของข่าวตามโครงสร้าง HTML ปัจจุบัน
        articles = driver.find_elements(By.XPATH, '//ul[@data-test="news-list"]/li//article[@data-test="article-item"]')
        for article in articles:
            try:
                # ดึงลิงก์ข่าวจาก <a> ที่มี data-test="article-title-link"
                link_element = article.find_element(By.XPATH, './/a[@data-test="article-title-link"]')
                partial_link = link_element.get_attribute('href')
                if partial_link:
                    cleaned_links.append(partial_link)
            except Exception as e:
                print(f"Error processing an article: {e}")
                continue
    except Exception as e:
        print(f"Error processing page: {e}")
        
    return np.unique(cleaned_links) if cleaned_links else []


In [45]:
all_company_urls = []
try:
    for page in range(1, 300):  
        try:
            print(f"Processing page {page}...")
            results = get_newslinks('nvidia-corp', page)
            if len(results) > 0:
                all_company_urls.extend(results)
                print(f"Found {len(results)} links on page {page}")
            else:
                print(f"No links found on page {page}")
            # หน่วงระหว่างการดึงแต่ละหน้าแบบสุ่ม
            time.sleep(random.uniform(2, 4))
        except Exception as e:
            print(f"Error on page {page}: {e}")
            # Reinitialize driver ถ้าจำเป็น
            driver.quit()
            driver = uc.Chrome(options=chrome_options)
finally:
    # ปิด driver เมื่อเสร็จสิ้นการทำงาน
    driver.quit()


Processing page 1...
Found 10 links on page 1
Processing page 2...
Found 10 links on page 2
Processing page 3...
Found 10 links on page 3
Processing page 4...
Found 10 links on page 4
Processing page 5...
Found 10 links on page 5
Processing page 6...
Found 10 links on page 6
Processing page 7...
Found 10 links on page 7
Processing page 8...
Found 10 links on page 8
Processing page 9...
Found 10 links on page 9
Processing page 10...
Found 10 links on page 10
Processing page 11...
Found 10 links on page 11
Processing page 12...
Found 10 links on page 12
Processing page 13...
Found 10 links on page 13
Processing page 14...
Found 10 links on page 14
Processing page 15...
Found 10 links on page 15
Processing page 16...
Found 10 links on page 16
Processing page 17...
Found 10 links on page 17
Processing page 18...
Found 10 links on page 18
Processing page 19...
Found 10 links on page 19
Processing page 20...
Found 10 links on page 20
Processing page 21...
Found 10 links on page 21
Processing

In [46]:
# สร้างฟังก์ชันสำหรับสร้าง ChromeOptions ใหม่
def get_chrome_options():
    options = uc.ChromeOptions()
    # ไม่ใช้ headless เพื่อให้ดูเหมือนผู้ใช้จริง (ถ้าต้องการแสดงหน้าต่างจริง)
    # options.add_argument('--headless')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36")
    options.add_argument("--disable-blink-features=AutomationControlled")
    return options

# สร้าง driver ครั้งแรกด้วย options ใหม่
driver = uc.Chrome(options=get_chrome_options())

import time
import random
import numpy as np
from bs4 import BeautifulSoup
from newspaper import Article
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from htmldate import find_date
from datetime import datetime
import re

def safe_find_datetime(url, html_content=None):
    """พยายามสกัดวันที่และเวลา จาก URL หรือ HTML content
       หากไม่พบจะคืนค่าวันที่และเวลาปัจจุบันเป็น fallback
    """
    # ลองใช้ find_date() จาก URL ก่อน (แต่ในกรณีของ Investing.com จะล้มเหลว)
    extracted_date = None
    try:
        if url:
            extracted_date = find_date(url)
    except Exception as e:
        print(f"URL date extraction failed: {e}")
    
    # หากมี HTML content ให้ลองสกัดด้วย regex
    if html_content:
        # ค้นหาข้อความที่มีรูปแบบ DD/MM/YYYY, HH:MM
        pattern = r'(\d{2}/\d{2}/\d{4}),\s*(\d{2}:\d{2})'
        match = re.search(pattern, html_content)
        if match:
            extracted_date = match.group(1)  # วันที่ในรูปแบบ DD/MM/YYYY
            extracted_time = match.group(2)  # เวลาในรูปแบบ HH:MM
            # แปลงวันที่เป็นรูปแบบที่ต้องการ (ถ้าต้องการ เช่น YYYY-MM-DD)
            try:
                dt_obj = datetime.strptime(extracted_date, "%d/%m/%Y")
                extracted_date = dt_obj.strftime("%Y-%m-%d")
            except Exception as e:
                print(f"Date conversion failed: {e}")
            return extracted_date, extracted_time

    # หากยังไม่พบ ให้ใช้ fallback เป็นวันที่และเวลาปัจจุบัน
    now = datetime.now()
    return now.strftime('%Y-%m-%d'), now.strftime('%H:%M')

# ตัวอย่าง DataFrame ที่เพิ่ม column "publish_time"
ticker = 'NVDA'
article_sentiments = pd.DataFrame({
    'ticker': [],
    'publish_date': [],
    'publish_time': [], 
    'title': [],
    'body_text': [],
    'url': [],
    'neg': [],
    'neu': [],
    'pos': [],
    'compound': []
})

# Process each URL (สมมุติ all_company_urls ถูกกำหนดไว้แล้ว)
for i, link in enumerate(all_company_urls):
    if i % 10 == 0:
        print(f"Processing article {i+1} of {len(all_company_urls)}")
    
    if not link or not isinstance(link, str):
        print(f"Invalid link: {link}")
        continue
        
    article = Article(link)
    
    try:
        driver.get(link)
    except Exception as e:
        print(f"Error accessing link {link}, reinitializing driver: {e}")
        try:
            driver.quit()
        except:
            pass
        driver = uc.Chrome(options=get_chrome_options())
        try:
            driver.get(link)
        except Exception as e:
            print(f"Still error accessing link {link}: {e}")
            continue

    time.sleep(random.uniform(2, 3))
    html = driver.page_source
    article.set_html(html)
    
    try:
        article.parse()
        text = article.text
        title = article.title
        if not text.strip():
            print(f"Empty text for {link}")
            continue
        if not title or title.strip() == "":
            title = "No title available"
    except Exception as e:
        print(f"Parsing failed for {link}: {e}")
        continue

    try:
        sid = SentimentIntensityAnalyzer()
        polarity = sid.polarity_scores(text)
    except Exception as e:
        print(f"Sentiment analysis failed: {e}")
        polarity = {'neg': 0.0, 'neu': 0.0, 'pos': 0.0, 'compound': 0.0}

    # ใช้ฟังก์ชัน safe_find_datetime เพื่อสกัดทั้งวันที่และเวลา
    publish_date, publish_time = safe_find_datetime(link, html)
    
    tmpdic = {
        'ticker': ticker,
        'publish_date': publish_date,
        'publish_time': publish_time,  # เพิ่ม column เวลา
        'title': title,
        'body_text': text, 
        'url': link,
        'neg': polarity['neg'],
        'neu': polarity['neu'],
        'pos': polarity['pos'],
        'compound': polarity['compound']
    }
    
    article_sentiments = pd.concat([article_sentiments, pd.DataFrame(tmpdic, index=[0])], ignore_index=True)

try:
    driver.quit()
    print("Driver closed successfully")
except:
    print("Driver was already closed or couldn't be closed")

print("Final DataFrame:")
print(f"Total articles collected: {len(article_sentiments)}")
print(article_sentiments.head())

Processing article 1 of 2990
Error accessing link https://uk.investing.com/news/economy-news/trump-auto-tariffs-announcement-lululemon-to-report--whats-moving-markets-3999772, reinitializing driver: HTTPConnectionPool(host='localhost', port=57308): Read timed out. (read timeout=120)
URL date extraction failed: ("URL couldn't be processed: %s", None)
URL date extraction failed: ("URL couldn't be processed: %s", None)
Error accessing link https://uk.investing.com/news/stock-market-news/amd-downgraded-nvidias-ai-chips-still-hold-a-significant-performance-advantage-3999872, reinitializing driver: HTTPConnectionPool(host='localhost', port=58909): Read timed out. (read timeout=120)
URL date extraction failed: ("URL couldn't be processed: %s", None)
URL date extraction failed: ("URL couldn't be processed: %s", None)
URL date extraction failed: ("URL couldn't be processed: %s", None)
URL date extraction failed: ("URL couldn't be processed: %s", None)
URL date extraction failed: ("URL couldn't 

In [47]:
# Show DataFrame of article sentiments data

article_sentiments

,ticker,publish_date,publish_time,title,body_text,url,neg,neu,pos,compound
0,NVDA,2025-03-27,08:50,"Trump auto tariffs announcement, Lululemon to ...",Investing.com - U.S. stock futures point to a ...,https://uk.investing.com/news/economy-news/tru...,0.045,0.877,0.078,0.9898
1,NVDA,2025-03-27,12:02,Wall Street analyst lists 3 reasons why you sh...,Risk Disclosure: Trading in financial instrume...,https://uk.investing.com/news/pro/wall-street-...,0.076,0.893,0.032,-0.8437
2,NVDA,2025-03-27,09:32,AMD downgraded: Nvidia’s AI chips still hold ’...,Investing.com -- Jefferies on Thursday downgra...,https://uk.investing.com/news/stock-market-new...,0.042,0.855,0.103,0.9704
3,NVDA,2025-03-27,09:52,Buy Nvidia shares as valuation is ’still compe...,Investing.com -- Bank of America (NYSE:) reaff...,https://uk.investing.com/news/stock-market-new...,0.045,0.854,0.101,0.9772
4,NVDA,2025-03-27,08:41,FTSE 100 Live: London blue-chips escape worst ...,"FTSE 100 falls 29 points to 8,661\n\nNext repo...",https://uk.investing.com/news/stock-market-new...,0.049,0.858,0.093,0.9996
...,...,...,...,...,...,...,...,...,...,...
2971,NVDA,2024-03-26,14:15,"March 26th, 2024 (Trade Strategy For SPY, QQQ,...","Benzinga - by RIPS, Benzinga Contributor.\n\nG...",https://uk.investing.com/news/stock-market-new...,0.033,0.863,0.104,0.9988
2972,NVDA,2024-03-26,12:50,Missed out on Nvidia and SMCI? Analyst says th...,"Given the artificial intelligence (AI) surge, ...",https://uk.investing.com/news/stock-market-new...,0.005,0.831,0.163,0.9977
2973,NVDA,2024-03-25,17:45,Tesla Bear? Nvidia Bull? Leveraged And Inverse...,"Benzinga - by Johnny Rice, Benzinga Staff Writ...",https://uk.investing.com/news/stock-market-new...,0.024,0.881,0.095,0.9928
2974,NVDA,2024-03-26,12:58,Tesla stock valuation 'still too high' says Be...,"In 2024, the ""Magnificent Seven,"" a group of t...",https://uk.investing.com/news/stock-market-new...,0.036,0.865,0.099,0.9884


In [48]:
all_company_urls

[np.str_('https://uk.investing.com/news/economy-news/trump-auto-tariffs-announcement-lululemon-to-report--whats-moving-markets-3999772'),
 np.str_('https://uk.investing.com/news/pro/wall-street-analyst-lists-3-reasons-why-you-should-buy-cisco-stock-432SI-4000318'),
 np.str_('https://uk.investing.com/news/stock-market-news/amd-downgraded-nvidias-ai-chips-still-hold-a-significant-performance-advantage-3999872'),
 np.str_('https://uk.investing.com/news/stock-market-news/buy-nvidia-shares-as-valuation-is-still-compelling-bank-of-america-says-3999895'),
 np.str_('https://uk.investing.com/news/stock-market-news/ftse-100-live-bluechip-stocks-tumble-on-tariff-hardballing-next-surges-to-new-high-3999768'),
 np.str_('https://uk.investing.com/news/stock-market-news/ftse-100-live-stocks-set-to-reverse-hard-after-trump-confirms-25-auto-tariffs-3999680'),
 np.str_('https://uk.investing.com/news/stock-market-news/nvidia-seen-stepping-in-to-prop-up-coreweaves-struggling-ipo-4001349'),
 np.str_('https:

In [ ]:
# Save DataFrame 

article_sentiments.to_pickle("nvda_article_sentiments_20210105.pkl")

In [50]:
article_sentiments.to_csv("nvda_article_sentiments_20250328.csv", sep=',', encoding='utf-8', header=True)

In [51]:
# Save URLS to text file

with open('nvda_urls_20210328.txt', 'w') as f:
    for link in all_company_urls:
        f.write("%s\n" % link)

In [20]:
import pandas as pd

article_sentiments = pd.read_pickle("nvda_article_sentiments_20210105.pkl")

In [21]:
# 1. แปลงคอลัมน์ publish_date เป็น datetime
article_sentiments['publish_date'] = pd.to_datetime(article_sentiments['publish_date'], errors='coerce')

# 2. แปลงคอลัมน์ publish_time เป็นเวลา (datetime.time)
article_sentiments['publish_time'] = pd.to_datetime(article_sentiments['publish_time'], format='%H:%M', errors='coerce').dt.time

article_sentiments['publish_datetime'] = pd.to_datetime(
    article_sentiments['publish_date'].astype(str) + ' ' + article_sentiments['publish_time'].astype(str),
    errors='coerce'
)


In [22]:
import re
from bs4 import BeautifulSoup

# สมมติว่า df คือ DataFrame ที่ได้จากการ scrape
# ตัวอย่างการโหลดไฟล์ pickle (ปรับ path ให้ตรงกับไฟล์ของคุณ)
# df = pd.read_pickle('/mnt/data/your_file.pkl')


def remove_html(text):
    if isinstance(text, str):
        return BeautifulSoup(text, "html.parser").get_text()
    return text

def remove_ads(text):
    if isinstance(text, str):
        # ลบข้อความที่มีรูปแบบโฆษณาที่พบได้บ่อย (สามารถปรับ regex ได้ตามความเหมาะสม)
        text = re.sub(r"here or remove ads.*?disclosureor", "", text, flags=re.IGNORECASE|re.DOTALL)
    return text

def clean_text(text):
    text = remove_html(text)      # ลบ HTML tags
    text = remove_ads(text)       # ลบข้อความโฆษณา
    text = text.strip()           # ตัดช่องว่างหัวและท้าย
    text = re.sub(r'[\r\n]+', ' ', text)
    text = re.sub(r'\s+', ' ', text)  # แทนที่ช่องว่างหลายตัวด้วยช่องว่างเดียว
    return text

for col in ['title', 'body_text']:
    article_sentiments[col] = article_sentiments[col].apply(clean_text)

article_sentiments = article_sentiments.drop_duplicates()

article_sentiments = article_sentiments.drop(columns=['publish_date', 'publish_time'])

C:\Users\acer\AppData\Local\Temp\ipykernel_2752\1103390801.py:11: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  return BeautifulSoup(text, "html.parser").get_text()


In [23]:
article_sentiments.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2975 entries, 0 to 2975
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   ticker            2975 non-null   object        
 1   title             2975 non-null   object        
 2   body_text         2975 non-null   object        
 3   url               2975 non-null   object        
 4   neg               2975 non-null   float64       
 5   neu               2975 non-null   float64       
 6   pos               2975 non-null   float64       
 7   compound          2975 non-null   float64       
 8   publish_datetime  2975 non-null   datetime64[ns]
dtypes: datetime64[ns](1), float64(4), object(4)
memory usage: 232.4+ KB


In [34]:
article_sentiments

,ticker,title,body_text,url,neg,neu,pos,compound,publish_datetime
0,NVDA,"Trump auto tariffs announcement, Lululemon to ...",Investing.com - U.S. stock futures point to a ...,https://uk.investing.com/news/economy-news/tru...,0.045,0.877,0.078,0.9898,2025-03-27 08:50:00
1,NVDA,Wall Street analyst lists 3 reasons why you sh...,Risk Disclosure: Trading in financial instrume...,https://uk.investing.com/news/pro/wall-street-...,0.076,0.893,0.032,-0.8437,2025-03-27 12:02:00
2,NVDA,AMD downgraded: Nvidia’s AI chips still hold ’...,Investing.com -- Jefferies on Thursday downgra...,https://uk.investing.com/news/stock-market-new...,0.042,0.855,0.103,0.9704,2025-03-27 09:32:00
3,NVDA,Buy Nvidia shares as valuation is ’still compe...,Investing.com -- Bank of America (NYSE:) reaff...,https://uk.investing.com/news/stock-market-new...,0.045,0.854,0.101,0.9772,2025-03-27 09:52:00
4,NVDA,FTSE 100 Live: London blue-chips escape worst ...,"FTSE 100 falls 29 points to 8,661 Next reports...",https://uk.investing.com/news/stock-market-new...,0.049,0.858,0.093,0.9996,2025-03-27 08:41:00
...,...,...,...,...,...,...,...,...,...
2971,NVDA,"March 26th, 2024 (Trade Strategy For SPY, QQQ,...","Benzinga - by RIPS, Benzinga Contributor. Good...",https://uk.investing.com/news/stock-market-new...,0.033,0.863,0.104,0.9988,2024-03-26 14:15:00
2972,NVDA,Missed out on Nvidia and SMCI? Analyst says th...,"Given the artificial intelligence (AI) surge, ...",https://uk.investing.com/news/stock-market-new...,0.005,0.831,0.163,0.9977,2024-03-26 12:50:00
2973,NVDA,Tesla Bear? Nvidia Bull? Leveraged And Inverse...,"Benzinga - by Johnny Rice, Benzinga Staff Writ...",https://uk.investing.com/news/stock-market-new...,0.024,0.881,0.095,0.9928,2024-03-25 17:45:00
2974,NVDA,Tesla stock valuation 'still too high' says Be...,"In 2024, the ""Magnificent Seven,"" a group of t...",https://uk.investing.com/news/stock-market-new...,0.036,0.865,0.099,0.9884,2024-03-26 12:58:00


In [35]:
article_sentiments = article_sentiments.sort_values(
    by='publish_datetime', 
    ascending=False
).reset_index(drop=True)

In [36]:
article_sentiments

,ticker,title,body_text,url,neg,neu,pos,compound,publish_datetime
0,NVDA,Nvidia seen stepping in to prop up CoreWeave’s...,"CoreWeave, an AI cloud service provider, faces...",https://uk.investing.com/news/stock-market-new...,0.054,0.850,0.096,0.9120,2025-03-27 17:16:00
1,NVDA,GPU catalysts? By Investing.com,Investing.com -- In a report published Thursda...,https://uk.investing.com/news/stock-market-new...,0.003,0.909,0.088,0.9882,2025-03-27 15:10:00
2,NVDA,Nvidia Will Anchor CoreWeave Deal at $40/sh Wi...,Risk Disclosure: Trading in financial instrume...,https://uk.investing.com/news/stock-market-new...,0.076,0.893,0.032,-0.8437,2025-03-27 13:48:00
3,NVDA,Wall Street analyst lists 3 reasons why you sh...,Risk Disclosure: Trading in financial instrume...,https://uk.investing.com/news/pro/wall-street-...,0.076,0.893,0.032,-0.8437,2025-03-27 12:02:00
4,NVDA,Buy Nvidia shares as valuation is ’still compe...,Investing.com -- Bank of America (NYSE:) reaff...,https://uk.investing.com/news/stock-market-new...,0.045,0.854,0.101,0.9772,2025-03-27 09:52:00
...,...,...,...,...,...,...,...,...,...
2970,NVDA,Stock Market Today: S&P500 closes lower ahead ...,Investing.com -- S&P 500 pared gains to close ...,https://uk.investing.com/news/stock-market-new...,0.053,0.858,0.089,0.9477,2024-03-26 00:40:00
2971,NVDA,Going all out on big tech may be dangerous as ...,Investing.com -- Big bets on big tech have bol...,https://uk.investing.com/news/stock-market-new...,0.024,0.805,0.171,0.9894,2024-03-25 20:24:00
2972,NVDA,Tesla Bear? Nvidia Bull? Leveraged And Inverse...,"Benzinga - by Johnny Rice, Benzinga Staff Writ...",https://uk.investing.com/news/stock-market-new...,0.024,0.881,0.095,0.9928,2024-03-25 17:45:00
2973,NVDA,SMCI stock surges 10% as JPMorgan starts at bu...,"Alongside Nvidia (NASDAQ:), Super Micro Comput...",https://uk.investing.com/news/stock-market-new...,0.012,0.869,0.119,0.9948,2024-03-25 11:22:00


In [37]:
article_sentiments.to_pickle("nvda_article_sentiments.pkl")